# Data Pipeline

This notebook runs the data pipeline for one coin in five steps:
1. Build daily RV from raw 5-minute data.
2. Build HAR inputs and rolling HAR forecasts.
3. Build LSTM residual inputs and rolling LSTM forecasts.
4. Build BNS-based daily jump indicators.
5. Build rolling Hawkes intensity features from the jump indicators.

 > Active coin: `BTC`

 > Outputs shown in this notebook:
- Step 1: `daily_rv`
- Step 2: dataset with `y_{t+1}`, `y_{t+1}^HAR`, and `y_{t+1}^HAR - y_{t+1}`
- Step 3: dataset with `date`, `y_t`, `y_t^HAR`, `e_t`, `e_t^LSTM`, and `y_t^LSTM`
- Step 4: dataset with daily BNS jump statistics and quality summary
- Step 5: dataset with `date`, `lambda_t`, and fitted Hawkes parameters for each rolling block

In [1]:
import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.pipeline.build_daily_rv as daily_rv_builder
import src.pipeline.HAR_forecast as har_forecast_builder
import src.pipeline.LSTM_forecast as lstm_forecast_builder

daily_rv_builder = importlib.reload(daily_rv_builder)
har_forecast_builder = importlib.reload(har_forecast_builder)
lstm_forecast_builder = importlib.reload(lstm_forecast_builder)

build_daily_rv = daily_rv_builder.build_daily_rv
load_har_config = har_forecast_builder.load_config
create_HAR_data = har_forecast_builder.create_HAR_data
run_har_rolling_forecast = har_forecast_builder.run_har_rolling_forecast

load_lstm_config = lstm_forecast_builder.load_config
create_LSTM_data = lstm_forecast_builder.create_LSTM_data
run_lstm_rolling_forecast = lstm_forecast_builder.run_lstm_rolling_forecast

COIN = "BTC"
CONFIG_PATH = PROJECT_ROOT / "config.yaml"

print("Data pipeline configuration loaded")
print(f"Coin: {COIN}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Config path: {CONFIG_PATH}")

Data pipeline configuration loaded
Coin: BTC
Project root: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modelling
Config path: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modelling\config.yaml


## Step 1. Build Daily RV

Build the daily realized volatility parquet for `BTC` from raw 5-minute Binance data using `build_daily_rv.py`.

In [2]:
import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.pipeline.build_daily_rv as daily_rv_builder

daily_rv_builder = importlib.reload(daily_rv_builder)
build_daily_rv = daily_rv_builder.build_daily_rv

COIN = "BTC"

daily_rv = build_daily_rv(coin=COIN)

print(f"Built daily RV for {COIN}")
display(daily_rv)

Raw file used: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modelling\data\raw\BTCUSDT_binance_5min_2017_2026.csv
Daily RV saved to: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modelling\data\interim\daily_rv\BTC.parquet
Data quality summary saved to: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modelling\results\data_quality\BTC\data_quality_summary.csv
Daily coverage details saved to: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modelling\results\data_quality\BTC\daily_coverage.csv
Valid interval for BTC: 2017-08-18 00:00:00+00:00 -> 2025-12-31 00:00:00+00:00
Rows written: 3041
Built daily RV for BTC


,RV
date,
2017-08-18 00:00:00+00:00,0.005784
2017-08-19 00:00:00+00:00,0.007632
2017-08-20 00:00:00+00:00,0.003788
2017-08-21 00:00:00+00:00,0.007837
2017-08-22 00:00:00+00:00,0.034540
...,...
2025-12-27 00:00:00+00:00,0.000040
2025-12-28 00:00:00+00:00,0.000054
2025-12-29 00:00:00+00:00,0.000435


## Step 2. Build HAR Dataset And Rolling Forecast

Use `HAR_forecast.py` to create the HAR dataset and run the rolling HAR forecast. Then display `y_{t+1}`, `y_{t+1}^HAR`, and `y_{t+1}^HAR - y_{t+1}`.

In [3]:
import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.pipeline.build_daily_rv as daily_rv_builder
import src.pipeline.HAR_forecast as har_forecast_builder

daily_rv_builder = importlib.reload(daily_rv_builder)
har_forecast_builder = importlib.reload(har_forecast_builder)

build_daily_rv = daily_rv_builder.build_daily_rv
load_har_config = har_forecast_builder.load_config
create_HAR_data = har_forecast_builder.create_HAR_data
run_har_rolling_forecast = har_forecast_builder.run_har_rolling_forecast

COIN = "BTC"
CONFIG_PATH = PROJECT_ROOT / "config.yaml"

# Step 2 is self-contained: ensure daily RV exists first.
_ = build_daily_rv(coin=COIN)

cfg_har = load_har_config(CONFIG_PATH)
har_dataset = create_HAR_data(coin=COIN, config=cfg_har)
har_forecast = run_har_rolling_forecast(coin=COIN, config=cfg_har)

har_display = har_forecast[["date", "y_true", "y_pred"]].copy()
har_display = har_display.rename(
    columns={
        "y_true": "y_{t+1}",
        "y_pred": "y_{t+1}^HAR",
    }
)
har_display["y_{t+1}^HAR - y_{t+1}"] = (
    har_display["y_{t+1}^HAR"] - har_display["y_{t+1}"]
)

print(f"Built HAR dataset for {COIN} with {len(har_dataset)} rows")
print(f"Built HAR forecast for {COIN} with {len(har_forecast)} rows")
display(har_display)

Raw file used: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modelling\data\raw\BTCUSDT_binance_5min_2017_2026.csv
Daily RV saved to: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modelling\data\interim\daily_rv\BTC.parquet
Data quality summary saved to: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modelling\results\data_quality\BTC\data_quality_summary.csv
Daily coverage details saved to: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modelling\results\data_quality\BTC\daily_coverage.csv
Valid interval for BTC: 2017-08-18 00:00:00+00:00 -> 2025-12-31 00:00:00+00:00
Rows written: 3041
HAR dataset built for BT

,date,y_{t+1},y_{t+1}^HAR,y_{t+1}^HAR - y_{t+1}
0,2020-06-24 00:00:00+00:00,-7.307577,-7.761575,-0.453998
1,2020-06-25 00:00:00+00:00,-7.506068,-7.623140,-0.117071
2,2020-06-26 00:00:00+00:00,-7.501154,-7.693687,-0.192533
3,2020-06-27 00:00:00+00:00,-8.364126,-7.633014,0.731112
4,2020-06-29 00:00:00+00:00,-8.714591,-8.013779,0.700812
...,...,...,...,...
2003,2025-12-23 00:00:00+00:00,-8.226310,-7.701805,0.524504
2004,2025-12-24 00:00:00+00:00,-8.642629,-7.999494,0.643136
2005,2025-12-25 00:00:00+00:00,-7.836804,-8.264572,-0.427769
2006,2025-12-26 00:00:00+00:00,-10.133532,-8.025440,2.108091


## Step 3. Build LSTM Dataset And Rolling Forecast

Use `LSTM_forecast.py` to create the LSTM residual dataset and run the rolling LSTM forecast. Then display `date`, `y_t`, `y_t^HAR`, `e_t`, `e_t^LSTM`, and `y_t^LSTM`.

In [4]:
import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.pipeline.build_daily_rv as daily_rv_builder
import src.pipeline.HAR_forecast as har_forecast_builder
import src.pipeline.LSTM_forecast as lstm_forecast_builder

daily_rv_builder = importlib.reload(daily_rv_builder)
har_forecast_builder = importlib.reload(har_forecast_builder)
lstm_forecast_builder = importlib.reload(lstm_forecast_builder)

build_daily_rv = daily_rv_builder.build_daily_rv
load_har_config = har_forecast_builder.load_config
run_har_rolling_forecast = har_forecast_builder.run_har_rolling_forecast

load_lstm_config = lstm_forecast_builder.load_config
create_LSTM_data = lstm_forecast_builder.create_LSTM_data
run_lstm_rolling_forecast = lstm_forecast_builder.run_lstm_rolling_forecast

COIN = "BTC"
CONFIG_PATH = PROJECT_ROOT / "config.yaml"

# Step 3 is self-contained: ensure prerequisites exist.
_ = build_daily_rv(coin=COIN)
cfg_har = load_har_config(CONFIG_PATH)
_ = run_har_rolling_forecast(coin=COIN, config=cfg_har)

cfg_lstm = load_lstm_config(CONFIG_PATH)
lstm_dataset = create_LSTM_data(coin=COIN, config=cfg_lstm)
lstm_forecast = run_lstm_rolling_forecast(coin=COIN, config=cfg_lstm)

lstm_display = lstm_forecast[["date", "y_t", "y_t^HAR", "e_t", "e_t^LSTM", "y_t^LSTM"]].copy()

print(f"Built LSTM dataset for {COIN} with {len(lstm_dataset)} rows")
print(f"Built LSTM forecast for {COIN} with {len(lstm_forecast)} rows")
display(lstm_display)

Raw file used: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modelling\data\raw\BTCUSDT_binance_5min_2017_2026.csv
Daily RV saved to: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modelling\data\interim\daily_rv\BTC.parquet
Data quality summary saved to: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modelling\results\data_quality\BTC\data_quality_summary.csv
Daily coverage details saved to: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modelling\results\data_quality\BTC\daily_coverage.csv
Valid interval for BTC: 2017-08-18 00:00:00+00:00 -> 2025-12-31 00:00:00+00:00
Rows written: 3041
HAR dataset built for BT

,date,y_t,y_t^HAR,e_t,e_t^LSTM,y_t^LSTM
0,2023-10-12 00:00:00+00:00,-8.493255,-8.548449,-0.055194,0.079699,-8.468750
1,2023-10-13 00:00:00+00:00,-10.258633,-8.460109,1.798525,0.079507,-8.380601
2,2023-10-14 00:00:00+00:00,-9.336595,-9.258993,0.077602,0.091628,-9.167365
3,2023-10-15 00:00:00+00:00,-5.576916,-8.903468,-3.326552,0.088232,-8.815236
4,2023-10-16 00:00:00+00:00,-7.829178,-7.150697,0.678481,0.067403,-7.083294
...,...,...,...,...,...,...
803,2025-12-23 00:00:00+00:00,-8.226310,-7.701805,0.524504,-0.791336,-8.493142
804,2025-12-24 00:00:00+00:00,-8.642629,-7.999494,0.643136,-0.041900,-8.041394
805,2025-12-25 00:00:00+00:00,-7.836804,-8.264572,-0.427769,-1.017774,-9.282346
806,2025-12-26 00:00:00+00:00,-10.133532,-8.025440,2.108091,1.764462,-6.260979


## Step 4. Build BNS-Based Jumps

Use `build_BNS_jumps.py` to compute daily BNS jump statistics from the raw 5-minute Binance data. Then display the daily jump dataframe and its quality summary.

In [3]:
import importlib
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.pipeline.build_BNS_jumps as bns_jump_builder

bns_jump_builder = importlib.reload(bns_jump_builder)
build_BNS_jumps = bns_jump_builder.build_BNS_jumps

COIN = "BTC"
output_dir = PROJECT_ROOT / "data" / "interim" / "daily_rv_jumps" / COIN
summary_file = output_dir / "data_quality_summary.csv"

bns_jump_df = build_BNS_jumps(coin=COIN)
bns_jump_df = bns_jump_df.copy()
bns_jump_df["date"] = pd.to_datetime(bns_jump_df["date"], errors="coerce")
bns_jump_df = bns_jump_df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

bns_quality_summary = pd.read_csv(summary_file)

print(f"Built BNS jump dataset for {COIN} with {len(bns_jump_df)} rows")
print(f"Jump quality summary path: {summary_file}")
display(bns_jump_df)
display(bns_quality_summary)

Daily RV with jumps saved to: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modelling\data\interim\daily_rv_jumps\BTC\RV_data.parquet
Data quality summary saved to: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modelling\data\interim\daily_rv_jumps\BTC\data_quality_summary.csv
Valid interval for BTC: 2017-08-18 00:00:00+00:00 -> 2025-12-31 00:00:00+00:00
Rows written: 3024
Raw file used: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modelling\data\raw\BTCUSDT_binance_5min_2017_2026.csv
Built BNS jump dataset for BTC with 3024 rows
Jump quality summary path: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modell

,date,RV,BPV,TPV,JV,z,jump_indicator,alpha,critical_value
0,2017-08-18 00:00:00+00:00,0.005784,0.004934,4.219477e-05,0.000850,2.428265,1,0.01,2.326348
1,2017-08-19 00:00:00+00:00,0.007632,0.006893,1.446495e-04,0.000739,1.206453,0,0.01,2.326348
2,2017-08-20 00:00:00+00:00,0.003788,0.002583,1.270074e-05,0.001205,5.013439,1,0.01,2.326348
3,2017-08-21 00:00:00+00:00,0.007837,0.007194,1.112727e-04,0.000644,1.217787,0,0.01,2.326348
4,2017-08-22 00:00:00+00:00,0.034540,0.029120,2.329892e-03,0.005421,2.058948,0,0.01,2.326348
...,...,...,...,...,...,...,...,...,...
3019,2025-12-27 00:00:00+00:00,0.000040,0.000037,2.296794e-09,0.000003,1.109718,0,0.01,2.326348
3020,2025-12-28 00:00:00+00:00,0.000054,0.000050,3.865242e-09,0.000004,1.146362,0,0.01,2.326348
3021,2025-12-29 00:00:00+00:00,0.000435,0.000399,3.094288e-07,0.000037,1.317595,0,0.01,2.326348
3022,2025-12-30 00:00:00+00:00,0.000313,0.000248,1.212680e-07,0.000065,3.214639,1,0.01,2.326348


,asset,expected_obs_per_day,coverage_threshold,jump_test_alpha,init_date,end_date,valid_start,valid_end,total_days_in_valid_interval,days_with_missing_data,pct_days_with_missing_data,days_completely_missed,pct_days_completely_missed,n_missing_dates_between,missing_dates_between,rows_saved,jump_days_detected
0,BTC,288,0.9,0.01,2017-08-18 00:00:00+00:00,2025-12-31 00:00:00+00:00,2017-08-17 00:00:00+00:00,2025-12-31 00:00:00+00:00,3059,35,1.144165,0,0.0,35,2017-08-17;2017-09-06;2017-12-04;2017-12-18;20...,3024,608


## Step 5. Build Hawkes Rolling Features

Use `build_Hawkes_feature.py` to fit the Hawkes model on rolling windows of daily jump indicators and compute the daily `lambda_t` feature for the following prediction block. Then display the resulting Hawkes feature dataframe.

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.pipeline.build_BNS_jumps as bns_jump_builder
import src.pipeline.build_Hawkes_feature as hawkes_feature_builder

bns_jump_builder = importlib.reload(bns_jump_builder)
hawkes_feature_builder = importlib.reload(hawkes_feature_builder)

build_BNS_jumps = bns_jump_builder.build_BNS_jumps
build_hawkes_feature = hawkes_feature_builder.build_hawkes_feature

COIN = "BTC"
output_file = PROJECT_ROOT / "data" / "interim" / "event_5_min" / COIN / "hawkes_feature.parquet"

# Step 5 is self-contained: ensure the jump dataset exists first.
_ = build_BNS_jumps(coin=COIN)
hawkes_feature_df = build_hawkes_feature(coin=COIN)

hawkes_feature_df = hawkes_feature_df.copy()
hawkes_feature_df["date"] = pd.to_datetime(hawkes_feature_df["date"], errors="coerce")
hawkes_feature_df = hawkes_feature_df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

print(f"Built Hawkes feature dataset for {COIN} with {len(hawkes_feature_df)} rows")
print(f"Hawkes feature path: {output_file}")
display(hawkes_feature_df)

Daily RV with jumps saved to: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modelling\data\interim\daily_rv_jumps\BTC\RV_data.parquet
Data quality summary saved to: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modelling\data\interim\daily_rv_jumps\BTC\data_quality_summary.csv
Valid interval for BTC: 2017-08-18 00:00:00+00:00 -> 2025-12-31 00:00:00+00:00
Rows written: 3024
Raw file used: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\LSTM-H for HAR residual modelling\data\raw\BTCUSDT_binance_5min_2017_2026.csv
Building Hawkes features for BTC: 253 rolling windows, window_size=1000, step_size=8
Processing Hawkes window 1/253 | train=[2017-08-18, 2020-06-05] | forecast=[2020-06-06, 2020-06-13] | events=268 | status=fitted
Process

,date,lambda_t,mu,alpha,beta,branching_ratio,window_start,window_end,n_events_window,scale_factor,fit_status,converged,neg_loglik
0,2020-06-06 00:00:00+00:00,1.262072,0.557083,0.120219,0.27258,0.441042,2017-08-18 00:00:00+00:00,2020-06-05 00:00:00+00:00,268,0.268,fitted,False,269.330044
1,2020-06-07 00:00:00+00:00,1.212407,0.557083,0.120219,0.27258,0.441042,2017-08-18 00:00:00+00:00,2020-06-05 00:00:00+00:00,268,0.268,fitted,False,269.330044
2,2020-06-08 00:00:00+00:00,1.166242,0.557083,0.120219,0.27258,0.441042,2017-08-18 00:00:00+00:00,2020-06-05 00:00:00+00:00,268,0.268,fitted,False,269.330044
3,2020-06-09 00:00:00+00:00,1.123328,0.557083,0.120219,0.27258,0.441042,2017-08-18 00:00:00+00:00,2020-06-05 00:00:00+00:00,268,0.268,fitted,False,269.330044
4,2020-06-10 00:00:00+00:00,1.083438,0.557083,0.120219,0.27258,0.441042,2017-08-18 00:00:00+00:00,2020-06-05 00:00:00+00:00,268,0.268,fitted,False,269.330044
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019,2025-12-27 00:00:00+00:00,0.894061,0.748245,0.131012,0.51440,0.254688,2023-03-30 00:00:00+00:00,2025-12-23 00:00:00+00:00,125,0.125,fitted,True,123.458854
2020,2025-12-28 00:00:00+00:00,0.884980,0.748245,0.131012,0.51440,0.254688,2023-03-30 00:00:00+00:00,2025-12-23 00:00:00+00:00,125,0.125,fitted,True,123.458854
2021,2025-12-29 00:00:00+00:00,0.876465,0.748245,0.131012,0.51440,0.254688,2023-03-30 00:00:00+00:00,2025-12-23 00:00:00+00:00,125,0.125,fitted,True,123.458854
2022,2025-12-30 00:00:00+00:00,0.868480,0.748245,0.131012,0.51440,0.254688,2023-03-30 00:00:00+00:00,2025-12-23 00:00:00+00:00,125,0.125,fitted,True,123.458854
